# Proyek Data Mining (ST167) — Master Notebook (v2)
## Studi Kasus: PT Retailindo Sejahtera — Multi-Agent Enterprise Assistant

**Versi ini langsung mulai dari 5 file CSV yang sudah bersih** (`clean_customer_service.csv`, `clean_billing_returns.csv`, `clean_sales_presales.csv`, `clean_inventory.csv`, `clean_marketing.csv`).

| Bagian | Isi |
|---|---|
| **Part 0** | Load 5 dataset bersih |
| **Part 2** | Embedding & Vector DB (ChromaDB) |
| **Part 3** | Multi-Agent System (LangChain/LangGraph) |
| **Part 4** | Fine-Tuning (LoRA) |
| **Part 5** | Evaluator Module |
| **Part 6** | Dashboard (Streamlit) |

**Cara pakai di Google Colab:** jalankan berurutan dari atas ke bawah (`Runtime > Run all`). Beberapa cell butuh kamu upload file / masukkan API key — instruksinya ada di tiap bagian.

**Rekomendasi hardware:** aktifkan GPU di `Runtime > Change runtime type > T4 GPU` sebelum mulai, terutama untuk Part 4 (fine-tuning).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd "/content/drive/MyDrive/PROYEK DATA MINING/UAS"

/content/drive/MyDrive/PROYEK DATA MINING/UAS


In [ ]:
# Install semua dependency yang dibutuhkan seluruh notebook
!pip install -q sentence-transformers chromadb langchain langchain-community langgraph langchain-groq peft transformers accelerate bitsandbytes datasets streamlit pyngrok

---
# PART 1 — Load Dataset Bersih

Upload 5 file CSV yang sudah dibersihkan:
- `clean_customer_service.csv`
- `clean_billing_returns.csv`
- `clean_sales_presales.csv`
- `clean_inventory.csv`
- `clean_marketing.csv`

In [ ]:
from google.colab import files
uploaded = files.upload()  # pilih ke-5 file clean_*.csv sekaligus

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

cs_df = pd.read_csv("clean_customer_service.csv")
billing_returns_df = pd.read_csv("clean_billing_returns.csv")
sales_presales_df = pd.read_csv("clean_sales_presales.csv")
inventory_df = pd.read_csv("clean_inventory.csv")
marketing_df = pd.read_csv("clean_marketing.csv")

print("Customer Service :", cs_df.shape)
print("Billing/Returns  :", billing_returns_df.shape)
print("Sales/Pre-Sales  :", sales_presales_df.shape)
print("Inventory        :", inventory_df.shape)
print("Marketing        :", marketing_df.shape)

print("\n=== PART 0 selesai — 5 dataframe siap: cs_df, billing_returns_df, sales_presales_df, inventory_df, marketing_df ===")

Customer Service : (2045, 16)
Billing/Returns  : (2035, 16)
Sales/Pre-Sales  : (410, 16)
Inventory        : (421570, 7)
Marketing        : (8190, 12)

=== PART 0 selesai — 5 dataframe siap: cs_df, billing_returns_df, sales_presales_df, inventory_df, marketing_df ===


---
# PART 2 — Embedding & Vector DB (ChromaDB)

Mengubah 5 dataframe dari Part 1 menjadi dokumen teks, membuat embedding, dan menyimpannya ke ChromaDB sebagai knowledge base RAG untuk tiap agent.

In [ ]:
def build_ticket_documents(df, division_name):
    docs, metadatas, ids = [], [], []
    for i, row in df.reset_index(drop=True).iterrows():
        text = (
            f"Subject: {row['subject']}\n"
            f"Customer message: {row['body']}\n"
            f"Agent response: {row['answer']}"
        )
        docs.append(text)
        metadatas.append({
            "division": division_name,
            "queue": str(row.get('queue', '')),
            "priority": str(row.get('priority', '')),
            "type": str(row.get('type', '')),
        })
        ids.append(f"{division_name}_{i}")
    return docs, metadatas, ids

cs_docs, cs_meta, cs_ids = build_ticket_documents(cs_df, "customer_service")
br_docs, br_meta, br_ids = build_ticket_documents(billing_returns_df, "billing_returns")
sp_docs, sp_meta, sp_ids = build_ticket_documents(sales_presales_df, "sales_presales")

print("Contoh dokumen Customer Service:\n")
print(cs_docs[0])

Contoh dokumen Customer Service:

Subject: Inquiry for In-Depth Details on Financial Institution Offerings
Customer message: Dear Customer Support Team,\n\nI hope this message reaches you in good health. I am writing to request detailed information about the spectrum of products provided by your financial institution. As a potential client, I am particularly eager to learn about the features, advantages, and terms linked to your investment and savings offerings.\n\nWould you be able to send comprehensive brochures or documentation that specify the details of your products? I am interested in information regarding account types, interest rates, fees, minimum deposit amounts, and any current promotional deals.
Agent response: Thank you for your interest in our financial products. We offer a diverse selection of investment and savings solutions tailored to various needs, including high-yield savings accounts, fixed-term deposits, mutual funds, and retirement plans. Each product features s

In [ ]:
def build_inventory_documents(df):
    agg = df.groupby(['Store', 'Dept', 'Type', 'Size']).agg(
        avg_weekly_sales=('Weekly_Sales', 'mean'),
        min_weekly_sales=('Weekly_Sales', 'min'),
        max_weekly_sales=('Weekly_Sales', 'max'),
        holiday_weeks=('IsHoliday', 'sum'),
        total_weeks=('Weekly_Sales', 'count'),
    ).reset_index()

    docs, metadatas, ids = [], [], []
    for i, row in agg.iterrows():
        text = (
            f"Store {int(row['Store'])} (Type {row['Type']}, size {int(row['Size'])} sq ft), "
            f"Department {int(row['Dept'])}: average weekly sales is "
            f"${row['avg_weekly_sales']:.2f}, ranging from ${row['min_weekly_sales']:.2f} "
            f"to ${row['max_weekly_sales']:.2f} across {int(row['total_weeks'])} weeks "
            f"({int(row['holiday_weeks'])} of which were holiday weeks)."
        )
        docs.append(text)
        metadatas.append({"division": "inventory", "store": int(row['Store']), "dept": int(row['Dept']), "store_type": row['Type']})
        ids.append(f"inventory_{int(row['Store'])}_{int(row['Dept'])}")
    return docs, metadatas, ids

inv_docs, inv_meta, inv_ids = build_inventory_documents(inventory_df)
print(f"Jumlah dokumen inventory: {len(inv_docs)}")
print(inv_docs[0])

Jumlah dokumen inventory: 3331
Store 1 (Type A, size 151315 sq ft), Department 1: average weekly sales is $22513.32, ranging from $14537.37 to $57592.12 across 143 weeks (10 of which were holiday weeks).


In [ ]:
def build_marketing_documents(df):
    agg = df.groupby('Store').agg(
        avg_temp=('Temperature', 'mean'), avg_fuel_price=('Fuel_Price', 'mean'),
        avg_markdown1=('MarkDown1', 'mean'), avg_markdown2=('MarkDown2', 'mean'),
        avg_markdown3=('MarkDown3', 'mean'), avg_markdown4=('MarkDown4', 'mean'),
        avg_markdown5=('MarkDown5', 'mean'), avg_cpi=('CPI', 'mean'),
        avg_unemployment=('Unemployment', 'mean'), holiday_weeks=('IsHoliday', 'sum'),
    ).reset_index()

    docs, metadatas, ids = [], [], []
    for i, row in agg.iterrows():
        total_markdown = sum(row[f'avg_markdown{k}'] for k in range(1, 6))
        text = (
            f"Store {int(row['Store'])} marketing profile: average total markdown/promotion "
            f"value is ${total_markdown:.2f} per week, average CPI is {row['avg_cpi']:.2f}, "
            f"average unemployment rate is {row['avg_unemployment']:.2f}%, "
            f"and there were {int(row['holiday_weeks'])} holiday weeks in the observed period."
        )
        docs.append(text)
        metadatas.append({"division": "marketing", "store": int(row['Store'])})
        ids.append(f"marketing_{int(row['Store'])}")
    return docs, metadatas, ids

mkt_docs, mkt_meta, mkt_ids = build_marketing_documents(marketing_df)
print(f"Jumlah dokumen marketing: {len(mkt_docs)}")
print(mkt_docs[0])

Jumlah dokumen marketing: 45
Store 1 marketing profile: average total markdown/promotion value is $10377.23 per week, average CPI is 217.84, average unemployment rate is 7.36%, and there were 13 holiday weeks in the observed period.


In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer('all-MiniLM-L6-v2')

def embed_documents(docs, batch_size=64):
    return embedder.encode(docs, batch_size=batch_size, show_progress_bar=True).tolist()

print("Model embedding siap. Dimensi vector:", embedder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model embedding siap. Dimensi vector: 384


/tmp/ipykernel_502/732143261.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Model embedding siap. Dimensi vector:", embedder.get_sentence_embedding_dimension())


In [ ]:
import chromadb

chroma_client = chromadb.PersistentClient(path="./chroma_db")

def create_and_populate_collection(name, docs, metadatas, ids):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    collection = chroma_client.create_collection(name=name)
    embeddings = embed_documents(docs)
    batch_size = 500
    for start in range(0, len(docs), batch_size):
        end = start + batch_size
        collection.add(documents=docs[start:end], embeddings=embeddings[start:end],
                        metadatas=metadatas[start:end], ids=ids[start:end])
    print(f"Collection '{name}' selesai: {collection.count()} dokumen tersimpan.")
    return collection

cs_collection = create_and_populate_collection("customer_service_kb", cs_docs, cs_meta, cs_ids)
br_collection = create_and_populate_collection("billing_returns_kb", br_docs, br_meta, br_ids)
sp_collection = create_and_populate_collection("sales_presales_kb", sp_docs, sp_meta, sp_ids)
inv_collection = create_and_populate_collection("inventory_kb", inv_docs, inv_meta, inv_ids)
mkt_collection = create_and_populate_collection("marketing_kb", mkt_docs, mkt_meta, mkt_ids)

print("\n=== PART 2 selesai — 5 collection ChromaDB siap dipakai untuk RAG ===")

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Collection 'customer_service_kb' selesai: 2045 dokumen tersimpan.


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Collection 'billing_returns_kb' selesai: 2035 dokumen tersimpan.


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Collection 'sales_presales_kb' selesai: 410 dokumen tersimpan.


Batches:   0%|          | 0/53 [00:00<?, ?it/s]

Collection 'inventory_kb' selesai: 3331 dokumen tersimpan.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Collection 'marketing_kb' selesai: 45 dokumen tersimpan.

=== PART 2 selesai — 5 collection ChromaDB siap dipakai untuk RAG ===


---
# PART 3 — Multi-Agent System (LangChain + LangGraph)

Membangun 4 agent spesialis (satu per divisi) + 1 Orchestrator Agent yang mengarahkan pertanyaan ke agent yang tepat, berinteraksi lewat LangGraph.

**LLM:** notebook ini pakai **Groq API** (gratis, cepat, model open source seperti Llama 3.3). Kamu bisa daftar API key gratis di https://console.groq.com/keys — atau ganti ke provider lain (OpenAI, dsb) dengan mengganti bagian inisialisasi LLM di bawah.

In [ ]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Masukkan GROQ_API_KEY: ")

Masukkan GROQ_API_KEY: ··········


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)
print("LLM siap:", llm.model_name)

LLM siap: llama-3.3-70b-versatile


## 3.1 — Retriever per Divisi (menghubungkan agent ke ChromaDB dari Part 2)

In [ ]:
def retrieve_context(collection, query, n_results=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=n_results)
    docs = results['documents'][0]
    return "\n\n".join(docs)

# Quick test
print(retrieve_context(cs_collection, "How do I reset my password?")[:300])

Subject: Detected Unauthorized Access
Customer message: Dear Customer Support, we are reporting a serious security incident involving unauthorized access to confidential medical data. The breach is suspected to have occurred due to weak password policies. Immediate action has been taken, including r


## 3.2 — Definisi Agent per Divisi

Setiap agent adalah fungsi yang: (1) ambil konteks relevan dari knowledge base miliknya (RAG), (2) minta LLM menjawab berdasarkan konteks itu.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

AGENT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Kamu adalah {agent_name} di perusahaan retail PT Retailindo Sejahtera. "
     "Jawab pertanyaan HANYA berdasarkan konteks berikut. "
     "Jika informasi tidak ada di konteks, katakan dengan jujur bahwa kamu tidak punya data untuk itu "
     "daripada mengarang jawaban.\n\nKonteks:\n{context}"),
    ("human", "{question}")
])

def make_agent(agent_name, collection):
    def agent_fn(question: str) -> dict:
        context = retrieve_context(collection, question)
        chain = AGENT_PROMPT | llm
        response = chain.invoke({"agent_name": agent_name, "context": context, "question": question})
        return {"agent": agent_name, "answer": response.content, "retrieved_context": context}
    return agent_fn

agent_customer_service = make_agent("Agent Customer Service", cs_collection)
agent_billing_returns = make_agent("Agent Finance & Retur", br_collection)
agent_inventory = make_agent("Agent Inventory & Store Operations", inv_collection)
agent_marketing = make_agent("Agent Marketing", mkt_collection)

print("4 agent divisi siap: agent_customer_service, agent_billing_returns, agent_inventory, agent_marketing")

4 agent divisi siap: agent_customer_service, agent_billing_returns, agent_inventory, agent_marketing


## 3.3 — Orchestrator Agent (LangGraph)

Orchestrator menentukan agent mana yang paling relevan untuk menjawab pertanyaan user, memanggilnya, dan bisa menggabungkan jawaban dari lebih dari satu agent kalau pertanyaannya lintas divisi.

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    question: str
    route: List[str]
    responses: List[dict]
    final_answer: str

ROUTER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Kamu adalah router/supervisor di PT Retailindo Sejahtera. Ada 4 agent divisi:\n"
     "- customer_service: pertanyaan umum, keluhan, akun\n"
     "- billing_returns: pertanyaan soal tagihan, pembayaran, retur, refund\n"
     "- inventory: pertanyaan soal stok, penjualan per toko/departemen\n"
     "- marketing: pertanyaan soal promosi, markdown, tren\n\n"
     "Balas HANYA dengan daftar nama agent yang relevan, dipisah koma (bisa lebih dari satu jika pertanyaan lintas divisi). "
     "Contoh output: customer_service,billing_returns"),
    ("human", "{question}")
])

def router_node(state: AgentState) -> AgentState:
    chain = ROUTER_PROMPT | llm
    result = chain.invoke({"question": state["question"]})
    route = [r.strip() for r in result.content.split(",") if r.strip() in
             ["customer_service", "billing_returns", "inventory", "marketing"]]
    if not route:
        route = ["customer_service"]  # fallback
    return {**state, "route": route}

AGENT_MAP = {
    "customer_service": agent_customer_service,
    "billing_returns": agent_billing_returns,
    "inventory": agent_inventory,
    "marketing": agent_marketing,
}

def agent_execution_node(state: AgentState) -> AgentState:
    responses = [AGENT_MAP[name](state["question"]) for name in state["route"]]
    return {**state, "responses": responses}

SYNTHESIS_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Gabungkan jawaban dari beberapa agent berikut menjadi satu jawaban akhir yang koheren untuk user. "
     "Jangan sebutkan proses internal, langsung jawab seperti satu asisten terpadu."),
    ("human", "Pertanyaan: {question}\n\nJawaban tiap agent:\n{agent_answers}")
])

def synthesis_node(state: AgentState) -> AgentState:
    agent_answers = "\n\n".join([f"[{r['agent']}]: {r['answer']}" for r in state["responses"]])
    chain = SYNTHESIS_PROMPT | llm
    result = chain.invoke({"question": state["question"], "agent_answers": agent_answers})
    return {**state, "final_answer": result.content}

graph = StateGraph(AgentState)
graph.add_node("router", router_node)
graph.add_node("execute_agents", agent_execution_node)
graph.add_node("synthesize", synthesis_node)

graph.set_entry_point("router")
graph.add_edge("router", "execute_agents")
graph.add_edge("execute_agents", "synthesize")
graph.add_edge("synthesize", END)

orchestrator = graph.compile()
print("Orchestrator (LangGraph) siap.")

Orchestrator (LangGraph) siap.


In [ ]:
# Uji coba end-to-end
result = orchestrator.invoke({"question": "Saya mau retur produk yang cacat, bagaimana prosesnya dan berapa lama refund cair?"})
print("Route yang dipilih:", result["route"])
print("\nJawaban akhir:\n", result["final_answer"])

Route yang dipilih: ['customer_service', 'billing_returns']

Jawaban akhir:
 Untuk melakukan retur produk yang cacat, Anda perlu mengikuti proses berikut:

1. Pastikan produk masih dalam kondisi asli dan kemasan, serta Anda memiliki struk pembelian.
2. Kunjungi portal retur online kami atau gunakan formulir kontak untuk memulai proses retur.
3. Anda akan diberikan alamat retur melalui portal, dan jika produk Anda memenuhi syarat, Anda akan mendapatkan label pengiriman pra-bayar; jika tidak, Anda perlu menanggung biaya pengiriman.
4. Setelah kami menerima produk yang dikembalikan, refund akan diproses dalam waktu 5-7 hari kerja dan akan dikreditkan ke metode pembayaran asli Anda.

Perlu diingat bahwa produk harus dikembalikan dalam waktu 30 hari sejak pembelian. Jika Anda memiliki pertanyaan lebih lanjut, silakan hubungi kami untuk bantuan lebih lanjut.


In [ ]:
# Uji coba pertanyaan lintas divisi
result2 = orchestrator.invoke({"question": "Kenapa penjualan toko 5 turun padahal lagi ada promo besar-besaran?"})
print("Route yang dipilih:", result2["route"])
print("\nJawaban akhir:\n", result2["final_answer"])

print("\n=== PART 3 selesai — orchestrator siap dipakai (variable: orchestrator) ===")

Route yang dipilih: ['inventory', 'marketing']

Jawaban akhir:
 Sayangnya, tidak ada informasi yang cukup untuk menjelaskan mengapa penjualan toko 5 turun meskipun ada promo besar-besaran. Data yang tersedia tidak menyediakan informasi tentang penurunan penjualan yang spesifik di toko 5 atau detail tentang promo yang diadakan. Oleh karena itu, tidak dapat diberikan jawaban pasti atas pertanyaan tersebut.

=== PART 3 selesai — orchestrator siap dipakai (variable: orchestrator) ===


---
# PART 4 — Fine-Tuning (LoRA)

Fine-tuning model kecil open source (`Qwen2.5-0.5B-Instruct`) memakai pasangan pertanyaan-jawaban dari data tiket, supaya model punya "gaya jawab" khas customer service PT Retailindo Sejahtera. Pakai **LoRA** (Low-Rank Adaptation) via library `peft` supaya ringan dan bisa jalan di GPU T4 gratis Colab.

Catatan: ini melengkapi RAG (Part 3) — RAG memberi *pengetahuan* faktual terkini, fine-tuning memberi *gaya/format* jawaban yang konsisten.

In [ ]:
import torch
from datasets import Dataset

# Gabungkan semua data tiket (CS + Billing/Returns) jadi satu training set instruction-following
def build_finetune_dataset(*dfs):
    records = []
    for df in dfs:
        for _, row in df.iterrows():
            records.append({
                "instruction": f"Subject: {row['subject']}\n\n{row['body']}",
                "output": row['answer']
            })
    return records

records = build_finetune_dataset(cs_df, billing_returns_df)
print("Total pasangan instruction-output:", len(records))
print(records[0])

Total pasangan instruction-output: 4080
{'instruction': 'Subject: Inquiry for In-Depth Details on Financial Institution Offerings\n\nDear Customer Support Team,\\n\\nI hope this message reaches you in good health. I am writing to request detailed information about the spectrum of products provided by your financial institution. As a potential client, I am particularly eager to learn about the features, advantages, and terms linked to your investment and savings offerings.\\n\\nWould you be able to send comprehensive brochures or documentation that specify the details of your products? I am interested in information regarding account types, interest rates, fees, minimum deposit amounts, and any current promotional deals.', 'output': 'Thank you for your interest in our financial products. We offer a diverse selection of investment and savings solutions tailored to various needs, including high-yield savings accounts, fixed-term deposits, mutual funds, and retirement plans. Each product f

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    messages = [
        {"role": "system", "content": "Kamu adalah agent customer service PT Retailindo Sejahtera. Jawab dengan sopan, jelas, dan solutif."},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

hf_dataset = Dataset.from_list(records)
hf_dataset = hf_dataset.map(format_example)

# Ambil subset supaya training cepat (bisa diperbesar kalau ada waktu/GPU lebih kuat)
hf_dataset = hf_dataset.shuffle(seed=42).select(range(min(2000, len(hf_dataset))))
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
print(hf_dataset)
print("\nContoh formatted text:\n", hf_dataset['train'][0]['text'][:500])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Map:   0%|          | 0/4080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'output', 'text'],
        num_rows: 1800
    })
    test: Dataset({
        features: ['instruction', 'output', 'text'],
        num_rows: 200
    })
})

Contoh formatted text:
 <|im_start|>system
Kamu adalah agent customer service PT Retailindo Sejahtera. Jawab dengan sopan, jelas, dan solutif.<|im_end|>
<|im_start|>user
Subject: Incompatible Software Issue with Systems

Customer Support team, I am reaching out to highlight a significant problem our company is currently experiencing. We are having difficulty analyzing data efficiently due to software incompatibilities. Despite multiple attempts, we have been unable to resolve the issue through software updates or reins


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [ ]:
def tokenize_fn(examples):
    tokens = tokenizer(examples["text"], truncation=True, max_length=512, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = hf_dataset.map(tokenize_fn, batched=True, remove_columns=hf_dataset["train"].column_names)

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./lora_finetuned_model",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.462939,0.435844
2,0.423946,0.416504


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=226, training_loss=0.6862168016686904, metrics={'train_runtime': 699.0976, 'train_samples_per_second': 5.149, 'train_steps_per_second': 0.323, 'total_flos': 3981984399360000.0, 'train_loss': 0.6862168016686904, 'epoch': 2.0})

In [ ]:
model.save_pretrained("./lora_finetuned_model")
tokenizer.save_pretrained("./lora_finetuned_model")
print("Model hasil fine-tuning tersimpan di ./lora_finetuned_model")

print("\n=== PART 4 selesai — model fine-tuned siap dipakai/dievaluasi ===")

Model hasil fine-tuning tersimpan di ./lora_finetuned_model

=== PART 4 selesai — model fine-tuned siap dipakai/dievaluasi ===


In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 44.1 MB/s eta 0:00:00


---
# PART 4.5 — Integrasi Model Fine-Tuned ke Agent

Bagian ini **menghubungkan** model hasil fine-tuning (Part 4) ke sistem multi-agent (Part 3) — supaya fine-tuning benar-benar dipakai di solusi, bukan cuma di-training lalu didiamkan.

**Strategi integrasi:**
- **Agent Customer Service** & **Agent Finance/Retur** → pakai **model fine-tuned lokal** (`./lora_finetuned_model`), karena data training-nya (Part 4) memang berasal dari tiket 2 divisi ini — jadi gaya jawabnya relevan.
- **Agent Inventory** & **Agent Marketing** → tetap pakai **Groq (Llama 3.3)**, karena datanya numerik/agregat dan tidak termasuk data training fine-tuning — memaksakan model fine-tuned di sini justru bisa menurunkan kualitas jawaban.

Ini pendekatan yang umum dipakai di produksi: fine-tuning dipakai selektif di area yang datanya representatif, bukan dipaksakan ke semua agent.

In [ ]:
from peft import PeftModel
from transformers import pipeline as hf_pipeline

# Load base model + adapter LoRA hasil Part 4, lalu gabungkan (merge) supaya inferensi lebih cepat
ft_tokenizer = AutoTokenizer.from_pretrained("./lora_finetuned_model")

ft_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)
ft_model = PeftModel.from_pretrained(ft_base_model, "./lora_finetuned_model")
ft_model = ft_model.merge_and_unload()

ft_text_pipeline = hf_pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3,
)

print("Model fine-tuned berhasil dimuat dan siap dipakai agent.")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model fine-tuned berhasil dimuat dan siap dipakai agent.


In [ ]:
def make_finetuned_agent(agent_name, collection):
    """Agent yang menjawab pakai model hasil fine-tuning (Part 4), bukan Groq."""
    def agent_fn(question: str) -> dict:
        context = retrieve_context(collection, question)
        messages = [
            {"role": "system", "content": (
                f"Kamu adalah {agent_name} di PT Retailindo Sejahtera. "
                f"Jawab pertanyaan berdasarkan konteks berikut dengan sopan, jelas, dan solutif. "
                f"Jika informasi tidak ada di konteks, katakan dengan jujur.\n\nKonteks:\n{context}"
            )},
            {"role": "user", "content": question},
        ]
        prompt = ft_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        output = ft_text_pipeline(prompt)[0]["generated_text"]
        answer = output[len(prompt):].strip()
        return {"agent": agent_name, "answer": answer, "retrieved_context": context}
    return agent_fn

# Ganti agent_customer_service dan agent_billing_returns dengan versi fine-tuned
agent_customer_service = make_finetuned_agent("Agent Customer Service (fine-tuned)", cs_collection)
agent_billing_returns = make_finetuned_agent("Agent Finance & Retur (fine-tuned)", br_collection)

# Perbarui AGENT_MAP di tempat (in-place) — orchestrator yang sudah di-compile di Part 3
# otomatis memakai fungsi baru ini karena AGENT_MAP diakses lewat referensi dict, bukan salinan
AGENT_MAP["customer_service"] = agent_customer_service
AGENT_MAP["billing_returns"] = agent_billing_returns

print("AGENT_MAP diperbarui:", {k: v.__name__ if hasattr(v, '__name__') else 'agent_fn' for k, v in AGENT_MAP.items()})

AGENT_MAP diperbarui: {'customer_service': 'agent_fn', 'billing_returns': 'agent_fn', 'inventory': 'agent_fn', 'marketing': 'agent_fn'}


In [ ]:
# Uji ulang orchestrator — sekarang customer_service & billing_returns pakai model fine-tuned
result_ft = orchestrator.invoke({"question": "Bagaimana cara reset password akun saya?"})
print("Route yang dipilih:", result_ft["route"])
print("\nJawaban akhir (dari model fine-tuned):\n", result_ft["final_answer"])

print("\n=== PART 4.5 selesai — fine-tuning terintegrasi ke agent, orchestrator memakai model fine-tuned untuk CS & Billing ===")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Route yang dipilih: ['customer_service']

Jawaban akhir (dari model fine-tuned):
 Untuk mengreset password Anda, Anda perlu mengakses situs web resmi Anda atau menggunakan aplikasi yang disediakan oleh sistem. Pastikan bahwa Anda telah melakukan login dengan username dan password yang benar. Setelah itu, cari opsi "Lupa Password" atau "Reset Password" dan ikuti instruksi yang diberikan untuk mengubah password Anda menjadi yang baru. Jika Anda masih mengalami kesulitan, Anda dapat menghubungi tim dukungan teknis untuk bantuan lebih lanjut.

=== PART 4.5 selesai — fine-tuning terintegrasi ke agent, orchestrator memakai model fine-tuned untuk CS & Billing ===


---
# PART 5 — Evaluator Module

Mengukur performa sistem multi-agent pada 5 dimensi sesuai soal UAS:
- **Accuracy** — kesesuaian jawaban dengan konteks/ground truth (pakai LLM-as-judge)
- **Effectiveness** — apakah masalah user benar-benar terjawab/terselesaikan
- **Efficiency** — waktu respons & estimasi jumlah token
- **Explainability** — apakah jawaban bisa dilacak ke sumber (dokumen RAG yang diambil)
- **Hallucination** — apakah jawaban mengarang info di luar konteks yang diambil

In [ ]:
import time
import json

EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Kamu adalah evaluator independen untuk sistem AI customer service. "
     "Nilai jawaban AI berikut berdasarkan konteks yang tersedia, dengan skor 1-5 (5=terbaik) untuk tiap dimensi:\n"
     "- accuracy: seberapa akurat jawaban dibanding konteks\n"
     "- effectiveness: seberapa efektif jawaban menyelesaikan masalah user\n"
     "- explainability: seberapa jelas jawaban bisa dilacak sumbernya dari konteks\n"
     "- hallucination: seberapa RENDAH tingkat karangan/informasi di luar konteks (5=tidak ada halusinasi sama sekali)\n\n"
     "Balas HANYA dalam format JSON valid, tanpa teks tambahan, contoh:\n"
     '{{"accuracy": 4, "effectiveness": 5, "explainability": 3, "hallucination": 5, "reasoning": "penjelasan singkat"}}'),
    ("human", "Pertanyaan: {question}\n\nKonteks yang tersedia:\n{context}\n\nJawaban AI:\n{answer}")
])


def latency_to_efficiency_score(latency_sec: float) -> int:
    """Ubah waktu respons jadi skor 1-5 (5=paling efisien), supaya Efficiency
    jadi dimensi skor eksplisit sejajar dengan 4 dimensi lain (sesuai soal UAS poin 4),
    bukan cuma angka detik mentah.

    Threshold ini bisa disesuaikan sesuai standar performa yang kalian tetapkan
    di laporan (di sini dipakai patokan umum untuk sistem RAG multi-agent).
    """
    if latency_sec < 2:
        return 5
    elif latency_sec < 4:
        return 4
    elif latency_sec < 6:
        return 3
    elif latency_sec < 8:
        return 2
    else:
        return 1


def evaluate_response(question, context, answer, response_latency_sec):
    start = time.time()
    chain = EVAL_PROMPT | llm
    result = chain.invoke({"question": question, "context": context, "answer": answer})
    eval_latency = time.time() - start

    try:
        scores = json.loads(result.content)
    except json.JSONDecodeError:
        scores = {"accuracy": None, "effectiveness": None, "explainability": None,
                   "hallucination": None, "reasoning": "gagal parse JSON dari evaluator"}

    # Efficiency dihitung dari waktu respons AGENT (bukan waktu evaluator menilai)
    scores["efficiency"] = latency_to_efficiency_score(response_latency_sec)
    scores["response_latency_sec"] = round(response_latency_sec, 2)
    scores["eval_latency_sec"] = round(eval_latency, 2)
    scores["estimated_tokens"] = len(answer.split())
    return scores

In [ ]:
def evaluate_agent_response(agent_response: dict, question: str, response_latency_sec: float) -> dict:
    """Evaluasi satu response agent (hasil dari make_agent() / make_finetuned_agent())."""
    scores = evaluate_response(question, agent_response["retrieved_context"], agent_response["answer"], response_latency_sec)
    scores["agent"] = agent_response["agent"]
    return scores

# --- Contoh evaluasi single-agent (termasuk Efficiency) ---
test_q = "Bagaimana cara mengajukan retur produk yang cacat?"

start = time.time()
resp = agent_billing_returns(test_q)
resp_latency = time.time() - start

eval_result = evaluate_agent_response(resp, test_q, resp_latency)

print("Jawaban agent:\n", resp["answer"][:300])
print("\nHasil evaluasi (termasuk Efficiency):\n", json.dumps(eval_result, indent=2))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Jawaban agent:
 Untuk membalas retu produk yang cacat, Anda bisa menghubungi kami melalui <tel_num>. Saya akan memberikan detail tentang prosedur pengiriman dan konsultasi untuk membantu Anda.

Hasil evaluasi (termasuk Efficiency):
 {
  "accuracy": 4,
  "effectiveness": 4,
  "explainability": 3,
  "hallucination": 4,
  "reasoning": "Jawaban AI sebagian besar akurat dan efektif, tetapi kurang jelas dalam menjelaskan prosedur retur produk yang cacat secara spesifik dan memiliki sedikit informasi di luar konteks yang diberikan.",
  "efficiency": 4,
  "response_latency_sec": 3.56,
  "eval_latency_sec": 0.65,
  "estimated_tokens": 24,
  "agent": "Agent Finance & Retur (fine-tuned)"
}


In [ ]:
def evaluate_orchestrator_run(question: str) -> dict:
    """Evaluasi hasil orchestrator penuh (multi-agent + sintesis), termasuk Efficiency."""
    start = time.time()
    result = orchestrator.invoke({"question": question})
    total_latency = time.time() - start

    combined_context = "\n\n".join([r["retrieved_context"] for r in result["responses"]])
    eval_scores = evaluate_response(question, combined_context, result["final_answer"], total_latency)
    eval_scores["route"] = result["route"]
    eval_scores["total_latency_sec"] = round(total_latency, 2)
    eval_scores["final_answer"] = result["final_answer"]
    return eval_scores

test_result = evaluate_orchestrator_run("Saya mau retur produk yang cacat, bagaimana prosesnya?")
print(json.dumps(test_result, indent=2, ensure_ascii=False))

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{
  "accuracy": 1,
  "effectiveness": 4,
  "explainability": 2,
  "hallucination": 2,
  "reasoning": "Jawaban AI tidak berdasarkan konteks yang tersedia, karena konteks tidak menyediakan informasi tentang proses pengembalian produk. Namun, jawaban AI masih memberikan solusi yang masuk akal dan efektif untuk menyelesaikan masalah pengembalian produk, meskipun tidak sepenuhnya berdasarkan informasi yang ada.",
  "efficiency": 1,
  "response_latency_sec": 11.72,
  "eval_latency_sec": 0.63,
  "estimated_tokens": 91,
  "route": [
    "billing_returns",
    "customer_service"
  ],
  "total_latency_sec": 11.72,
  "final_answer": "Untuk mengembalikan produk yang cacat, kami memerlukan informasi lebih lanjut tentang produk tersebut dan masalah yang Anda hadapi. Silakan berikan detail tentang kesalahan atau cacat yang Anda temukan, termasuk kapan dan di mana kesalahan itu terjadi, serta kemungkinan penyebabnya. Ini akan membantu kami memahami situasi dengan lebih baik dan memberikan solusi yang 

In [ ]:
# Evaluasi batch — jalankan beberapa pertanyaan uji sekaligus, hasilnya jadi tabel ringkasan
# Sekarang mencakup SEMUA 5 dimensi sesuai soal UAS poin 4:
# Accuracy, Effectiveness, Efficiency, Explainability, Hallucination
test_questions = [
    "Bagaimana cara reset password akun saya?",
    "Saya mau retur produk yang cacat, bagaimana prosesnya?",
    "Toko mana yang penjualannya paling tinggi bulan ini?",
    "Kenapa promosi minggu ini kurang efektif?",
    "Berapa lama proses refund biasanya cair?",
]

eval_records = []
for q in test_questions:
    r = evaluate_orchestrator_run(q)
    eval_records.append({
        "question": q,
        "route": ", ".join(r.get("route", [])),
        "accuracy": r.get("accuracy"),
        "effectiveness": r.get("effectiveness"),
        "efficiency": r.get("efficiency"),
        "explainability": r.get("explainability"),
        "hallucination": r.get("hallucination"),
        "latency_sec": r.get("total_latency_sec"),
    })

eval_df = pd.DataFrame(eval_records)
eval_df.to_csv("evaluation_results.csv", index=False)
print(eval_df)
print("\nRata-rata skor (5 dimensi):\n",
      eval_df[["accuracy", "effectiveness", "efficiency", "explainability", "hallucination"]].mean())

print("\n=== PART 5 selesai — evaluator mencakup 5 dimensi lengkap (Accuracy, Effectiveness, Efficiency, Explainability, Hallucination) ===")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

                                            question  \
0           Bagaimana cara reset password akun saya?   
1  Saya mau retur produk yang cacat, bagaimana pr...   
2  Toko mana yang penjualannya paling tinggi bula...   
3          Kenapa promosi minggu ini kurang efektif?   
4           Berapa lama proses refund biasanya cair?   

                               route  accuracy  effectiveness  efficiency  \
0                   customer_service         3              2           4   
1  customer_service, billing_returns         2              4           3   
2                          inventory         5              4           5   
3               marketing, inventory         5              4           4   
4                    billing_returns         3              4           5   

   explainability  hallucination  latency_sec  
0               2              4         3.20  
1               2              4         5.50  
2               5              5         1.40  
3       

---
# PART 6 — Dashboard (Streamlit)

Dashboard tidak berjalan di dalam notebook — Streamlit adalah aplikasi web terpisah. Cell di bawah ini akan **menulis file `app.py`**, lalu menjalankannya sebagai proses terpisah dan mengekspos ke internet lewat `ngrok` (supaya bisa diakses dari Colab, yang tidak punya localhost publik).

**Untuk deployment final** (upload ke launchinpad.amikom.ac.id), jalankan `app.py` ini di server/laptop sendiri dengan `streamlit run app.py` — tidak perlu ngrok kalau sudah di server publik.

## Menjalankan Dashboard dari Colab (opsional, pakai ngrok)

Kalau mau demo langsung dari Colab tanpa deploy ke server, jalankan cell berikut. Butuh akun ngrok gratis (https://ngrok.com) untuk dapat authtoken.

In [ ]:
!pip install -q plotly

In [ ]:
from pyngrok import ngrok
import getpass, os

if "NGROK_AUTHTOKEN" not in os.environ:
    os.environ["NGROK_AUTHTOKEN"] = getpass.getpass("Masukkan ngrok authtoken: ")
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])

get_ipython().system_raw("streamlit run app.py --server.port 8501 &")
public_url = ngrok.connect(8501)
print("Dashboard bisa diakses di:", public_url)

Dashboard bisa diakses di: NgrokTunnel: "https://unstamped-outnumber-schilling.ngrok-free.dev" -> "http://localhost:8501"


---
## Ringkasan Akhir Project

| Part | Status | Output |
|---|---|---|
| 0. Load Dataset Bersih | ✅ | 5 dataframe (dari clean_*.csv) |
| 2. Embedding & Vector DB | ✅ | ChromaDB dengan 5 collection |
| 3. Multi-Agent System | ✅ | `orchestrator` (LangGraph) |
| 4. Fine-Tuning | ✅ | Model LoRA di `./lora_finetuned_model` |
| **4.5. Integrasi Fine-Tuning ke Agent** | ✅ | Agent CS & Billing/Retur pakai model fine-tuned; Agent Inventory & Marketing tetap pakai Groq |
| 5. Evaluator | ✅ | `evaluation_results.csv` — **5 dimensi lengkap**: Accuracy, Effectiveness, **Efficiency**, Explainability, Hallucination |
| 6. Dashboard | ✅ | `app.py` (Streamlit) |

**Kesesuaian dengan soal UAS poin 3 (60 poin):** keempat pendekatan (Fine-tuning, RAG, Embedding, Vector DB) sekarang **terintegrasi dalam satu solusi**, bukan berdiri sendiri — fine-tuning benar-benar dipakai oleh 2 dari 4 agent, bukan cuma di-training lalu tidak terpakai.

**Kesesuaian dengan soal UAS poin 4 (20 poin):** evaluator sekarang mencakup **5 dari 5 dimensi** yang diminta soal (Accuracy, Effectiveness, Efficiency, Explainability, Hallucination), masing-masing sebagai skor 1-5.

**Untuk deployment final ke launchinpad.amikom.ac.id:**
1. `pipeline.py` (sudah terpisah dari notebook) berisi seluruh logika backend — pastikan `pipeline.py` juga diperbarui dengan perubahan Part 4.5 & Efficiency (lihat file `pipeline.py` versi terbaru)
2. Pastikan `chroma_db/` dan `lora_finetuned_model/` ikut di-deploy bersama kode
3. Simpan API key (Groq/ngrok) sebagai environment variable, jangan hardcode di kode
4. Jalankan `streamlit run app.py` di server, lalu upload sesuai instruksi launchinpad
5. Lengkapi laporan tertulis (studi kasus → arsitektur → hasil evaluasi) sebelum submit